# Part 2C: LVK populations and PE checks

**FQCP 2026 · Bayesian parameter estimation for gravitational-wave sources**

> Google Colab worksheet for early-stage graduate students. Run from top to
> bottom. In the JupyterBook, **Live route** cards identify the material for the
> session; **Extension** sections may be skipped live.

## Goal and route

Turn event-level information into a population statement and see why the detected catalogue is not the underlying population.

:::{admonition} Live route
:class: tip

Run the selection-bias laboratory, then complete the question cell.
:::


**Boundary:** The toy treats event masses as exactly measured. Production population inference reweights uncertain event posterior samples and estimates selection with injection campaigns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

rng = np.random.default_rng(20260817)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. From events to a population

Event-level posteriors become inputs to hierarchical inference. If $\Lambda$ describes a population,
$$
p(\Lambda\mid\{d_i\},\mathrm{det})\propto p(\Lambda)
\prod_i\frac{\int p(d_i\mid\theta)p(\theta\mid\Lambda)d\theta}{\alpha(\Lambda)}.
$$
$\alpha(\Lambda)$ is the detectable fraction. Ignoring it confuses the observed catalogue with the astrophysical population.

In [ ]:
from scipy.stats import norm

population_mean, population_width = 28.0, 5.0
all_masses = rng.normal(population_mean, population_width, 8000)
all_masses = all_masses[(all_masses > 8) & (all_masses < 55)]


def detection_probability(mass):
    return 1 / (1 + np.exp(-(mass - 22) / 3.5))


detected = all_masses[rng.random(all_masses.size) < detection_probability(all_masses)][
    :40
]
mean_grid = np.linspace(18, 38, 320)
integration_grid = np.linspace(8, 55, 900)
naive = []
corrected = []
for mean in mean_grid:
    event_term = norm.logpdf(detected, mean, population_width).sum()
    alpha = np.trapezoid(
        norm.pdf(integration_grid, mean, population_width)
        * detection_probability(integration_grid),
        integration_grid,
    )
    naive.append(event_term)
    corrected.append(event_term - len(detected) * np.log(alpha))


def normalise_population(logp):
    p = np.exp(logp - np.max(logp))
    return p / np.trapezoid(p, mean_grid)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
mass_axis = np.linspace(8, 55, 300)
axes[0].hist(all_masses, bins=35, density=True, histtype="step", label="underlying")
axes[0].hist(detected, bins=13, density=True, alpha=0.5, label="detected")
axes[0].plot(
    mass_axis, detection_probability(mass_axis) / 20, "--", label="selection (scaled)"
)
axes[0].set(
    xlabel="mass [toy units]", ylabel="density", title="Detected is not underlying"
)
axes[0].legend()
axes[1].plot(
    mean_grid, normalise_population(np.array(naive)), label="ignores selection"
)
axes[1].plot(
    mean_grid, normalise_population(np.array(corrected)), label="selection-aware"
)
axes[1].axvline(population_mean, color="k", ls="--", label="injection")
axes[1].set(
    xlabel="population mean",
    ylabel="posterior density",
    title="Selection changes the answer",
)
axes[1].legend()
plt.show()

In [ ]:
print(f"Injected population mean: {population_mean:.2f}")
print(f"Detected-catalogue mean: {detected.mean():.2f}")
print(f"Naive MAP: {mean_grid[np.argmax(naive)]:.2f}")
print(f"Selection-aware MAP: {mean_grid[np.argmax(corrected)]:.2f}")

This compact example treats masses as exactly measured. Real population inference reweights uncertain event posteriors, estimates selection with injection campaigns, infers several hyperparameters and often the rate, and checks sensitivity to event-level priors and waveform systematics.

## PE checks before population claims

- Inspect sampler convergence and effective sample size at event level.
- Check prior support and whether posterior samples press against boundaries.
- Perform posterior-predictive and residual checks.
- Reweight using the actual event-level sampling priors.
- Verify that injections represent the analysed population and detection pipeline.
- Repeat under plausible waveform, calibration, and population-model alternatives.